In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Parametri utilizați frecvent pentru transpilare

<details>
<summary><b>Versiuni de pachete</b></summary>

Codul de pe această pagină a fost dezvoltat folosind următoarele cerințe.
Recomandăm utilizarea acestor versiuni sau a unora mai noi.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Această pagină descrie unii dintre parametrii utilizați mai frecvent pentru transpilarea locală. Acești parametri sunt configurați folosind argumente pentru [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) sau [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile).

<span id="approx-degree"></span>
## Gradul de aproximare
Poți folosi gradul de aproximare pentru a specifica cât de mult dorești ca circuitul rezultat să corespundă circuitului dorit (de intrare). Acesta este un număr de tip float în intervalul (0.0 - 1.0), unde 0.0 reprezintă aproximarea maximă și 1.0 (implicit) înseamnă fără aproximare. Valorile mai mici sacrifică acuratețea ieșirii în favoarea ușurinței de execuție (adică, mai puține porți). Valoarea implicită este 1.0.

În sinteza unitară cu doi qubiți (utilizată în etapele inițiale ale tuturor nivelurilor și pentru etapa de optimizare cu nivelul de optimizare 3), această valoare specifică fidelitatea țintă a descompunerii de ieșire. Adică, câtă eroare este introdusă atunci când o reprezentare matriceală a unui circuit este convertită în porți discrete. Dacă gradul de aproximare are o valoare mai mică (mai multă aproximare), circuitul de ieșire din sinteză va diferi mai mult față de matricea de intrare, dar va avea și mai puține porți (deoarece orice operație arbitrară cu doi qubiți poate fi descompusă perfect cu cel mult trei porți CX) și este mai ușor de rulat.

Când gradul de aproximare este mai mic de 1.0, pot fi sintetizate circuite cu una sau două porți CX, ducând la mai puțină eroare de la hardware, dar mai multă din aproximare. Deoarece CX este cea mai costisitoare poartă în termeni de eroare, ar putea fi benefic să reduci numărul acestora cu prețul fidelității în sinteză (această tehnică a fost folosită pentru a crește volumul cuantic pe dispozitivele IBM&reg;: [Validating quantum computers using randomized model circuits](https://arxiv.org/abs/1811.12926)).

Ca exemplu, generăm un `UnitaryGate` aleatoriu cu doi qubiți care va fi sintetizat în etapa inițială. Setând `approximation_degree` la o valoare mai mică de 1.0 se poate genera un circuit aproximativ. Trebuie, de asemenea, să specificăm `basis_gates` pentru a permite metodei de sinteză să știe ce porți poate folosi pentru sinteza aproximativă.

In [1]:
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import random_unitary
from qiskit.transpiler import generate_preset_pass_manager

UU = random_unitary(4, seed=12345)
rand_U = UnitaryGate(UU)

qubits = QuantumRegister(2, name="q")
qc = QuantumCircuit(qubits)
qc.append(rand_U, qubits)
pass_manager = generate_preset_pass_manager(
    optimization_level=1,
    approximation_degree=0.85,
    basis_gates=["sx", "rz", "cx"],
)
approx_qc = pass_manager.run(qc)
print(approx_qc.count_ops()["cx"])

2


This yields an output of `2` because the approximation requires fewer CX gates.

<span id="seed"></span>
## Random number generator seed

Some parts of the transpiler are stochastic, so repeated transpilation runs may return different results. To obtain a reproducible result, you can set the seed for the pseudorandom number generator using the `seed_transpiler` argument. Repeated runs using the same seed will return the same results.

Example:

In [2]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, seed_transpiler=11, basis_gates=["sx", "rz", "cx"]
)
optimized_1 = pass_manager.run(qc)
optimized_1.draw("mpl")

<Image src="../docs/images/guides/common-parameters/extracted-outputs/dbc652e8-53a4-47a9-a66e-d9c1e5ef07c9-0.svg" alt="Output of the previous code cell" />

Aceasta produce o ieșire de `2` deoarece aproximarea necesită mai puține porți CX.

<span id="seed"></span>
## Seed-ul generatorului de numere aleatorii
Unele părți ale Transpiler-ului sunt stocastice, astfel încât rulările repetate ale transpilării pot returna rezultate diferite. Pentru a obține un rezultat reproductibil, poți seta seed-ul pentru generatorul de numere pseudoaleatorii folosind argumentul `seed_transpiler`. Rulările repetate cu același seed vor returna aceleași rezultate.

Exemplu:

In [3]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit.transpiler import Layout

backend = FakeSherbrooke()

a, b = qubits
initial_layout = Layout({a: 5, b: 6})

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, initial_layout=initial_layout
)
transpiled_circ = pass_manager.run(qc)

transpiled_circ.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/common-parameters/extracted-outputs/e18c034c-eb26-4d9d-81d7-37e0eafa17c7-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/dbc652e8-53a4-47a9-a66e-d9c1e5ef07c9-0.svg)

<span id="init-layout"></span>
## Layout-ul inițial
Înainte de transpilare, qubiții conținuți în circuitul tău sunt qubiți virtuali care nu corespund neapărat qubiților fizici de pe Backend-ul țintă. Poți specifica maparea inițială a qubiților virtuali la qubiții fizici folosind argumentul `initial_layout`. Reține că layout-ul final al qubiților poate diferi de layout-ul inițial deoarece Transpiler-ul poate permuta qubiții folosind porți swap sau alte mijloace.

În exemplul de mai jos, construim un layout inițial pentru Backend-ul simulat [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke) prin crearea unui obiect [`Layout`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.Layout). Layout-ul nostru mapează primul qubit din circuitul nostru la qubitul 5 al lui Sherbrooke, și mapează al doilea qubit din circuitul nostru la qubitul 6 al lui Sherbrooke. Reține că qubiții fizici sunt întotdeauna reprezentați prin numere întregi.

In [4]:
initial_layout = [5, 6]

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, initial_layout=initial_layout
)
transpiled_circ = pass_manager.run(qc)

transpiled_circ.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/common-parameters/extracted-outputs/a7800d8a-7354-48e4-a55f-f902ae28c875-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/e18c034c-eb26-4d9d-81d7-37e0eafa17c7-0.svg)

Pe lângă specificarea unui obiect Layout, poți transmite și o listă de numere întregi, unde elementul $i$ al listei conține qubitul fizic la care ar trebui mapat qubitul $i$. De exemplu:

In [5]:
from qiskit.visualization import plot_error_map

plot_error_map(backend, figsize=(30, 24))

<Image src="../docs/images/guides/common-parameters/extracted-outputs/8df57c6a-1ff4-4d58-9b7e-4378452c3025-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/a7800d8a-7354-48e4-a55f-f902ae28c875-0.svg)

Poți folosi funcția [`plot_error_map`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.plot_error_map) pentru a genera o diagramă a grafului dispozitivului cu informații despre erori și cu qubiții fizici etichetați. Poți vizualiza, de asemenea, diagrame similare pe pagina [Compute resources](https://quantum.cloud.ibm.com/computers).